In [ ]:
!pip install -q qiskit qiskit-machine-learning kagglehub torchvision

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
from PIL import Image

from kagglehub import dataset_download
from sklearn.metrics import classification_report, confusion_matrix

from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector
from qiskit import QuantumCircuit

In [ ]:
# -----------------------
# Config
# -----------------------
IMAGE_SIZE = 224
BATCH_SIZE = 16
FEATURE_DIM = 8          # keep ≤ 8
EPOCHS = 20
# EPOCHS_STAGE2 = 20
LR= 1e-4
# LR_STAGE2 = 1e-5

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
dataset_root = dataset_download("aryashah2k/breast-ultrasound-images-dataset")
print("dataset_root:", dataset_root)

data_root = os.path.join(dataset_root, "Dataset_BUSI_with_GT")
print("Folders:", os.listdir(data_root))

Using Colab cache for faster access to the 'breast-ultrasound-images-dataset' dataset.
dataset_root: /kaggle/input/breast-ultrasound-images-dataset
Folders: ['benign', 'normal', 'malignant']


In [ ]:
# -----------------------
# Dataset (mask-safe)
# Binary label: malignant=1, benign/normal=0
# -----------------------
class BUSIBinaryDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        self.samples = []
        self.class_names = ["non_malignant", "malignant"]  # 0,1

        for cls in ["benign", "normal", "malignant"]:
            cls_path = os.path.join(root_dir, cls)
            if not os.path.isdir(cls_path):
                continue

            for fname in os.listdir(cls_path):
                f = fname.lower()

                if not f.endswith(".png"):
                    continue
                if f.endswith("_mask.png"):
                    continue

                label = 1 if cls == "malignant" else 0
                self.samples.append((os.path.join(cls_path, fname), label))

        if len(self.samples) == 0:
            raise RuntimeError(f"No valid images found in {root_dir}")

        print(f"Loaded {len(self.samples)} images (masks excluded)")
        # quick counts
        n0 = sum(1 for _, y in self.samples if y == 0)
        n1 = sum(1 for _, y in self.samples if y == 1)
        print(f"Counts -> non_malignant: {n0}, malignant: {n1}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(label, dtype=torch.float32)  # float for BCE

In [ ]:
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std=[0.229,0.224,0.225])
])

full_dataset = BUSIBinaryDataset(data_root, transform=transform)

n = len(full_dataset)
if n < 2:
    raise ValueError(f"Dataset too small to split: {n} samples")

train_size = int(0.8 * n)
val_size   = n - train_size

generator = torch.Generator().manual_seed(42)
train_ds, val_ds = random_split(full_dataset, [train_size, val_size], generator=generator)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train samples: {len(train_ds)} | Val samples: {len(val_ds)}")
print("Binary classes:", full_dataset.class_names)

Loaded 798 images (masks excluded)
Counts -> non_malignant: 587, malignant: 211
Train samples: 638 | Val samples: 160
Binary classes: ['non_malignant', 'malignant']


In [ ]:
feature_map = ZZFeatureMap(feature_dimension=FEATURE_DIM, reps=1)
ansatz = RealAmplitudes(num_qubits=FEATURE_DIM, reps=1)

qc = QuantumCircuit(FEATURE_DIM)
qc.compose(feature_map, inplace=True)
qc.compose(ansatz, inplace=True)

qnn = EstimatorQNN(
    circuit=qc,
    input_params=feature_map.parameters,
    weight_params=ansatz.parameters
)

/tmp/ipython-input-86803684.py:1: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(feature_dimension=FEATURE_DIM, reps=1)
/tmp/ipython-input-86803684.py:2: DeprecationWarning: The class ``qiskit.circuit.library.n_local.real_amplitudes.RealAmplitudes`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.real_amplitudes instead.
  ansatz = RealAmplitudes(num_qubits=FEATURE_DIM, reps=1)


In [ ]:
qc.decompose().draw()

┌───┐┌───────────┐                                             »
q_0: ┤ H ├┤ P(2*x[0]) ├──■──────────────────────────────────■────■──»
     ├───┤├───────────┤┌─┴─┐┌────────────────────────────┐┌─┴─┐  │  »
q_1: ┤ H ├┤ P(2*x[1]) ├┤ X ├┤ P(2*(π - x[0])*(π - x[1])) ├┤ X ├──┼──»
     ├───┤├───────────┤└───┘└────────────────────────────┘└───┘┌─┴─┐»
q_2: ┤ H ├┤ P(2*x[2]) ├────────────────────────────────────────┤ X ├»
     ├───┤├───────────┤                                        └───┘»
q_3: ┤ H ├┤ P(2*x[3]) ├─────────────────────────────────────────────»
     ├───┤├───────────┤                                             »
q_4: ┤ H ├┤ P(2*x[4]) ├─────────────────────────────────────────────»
     ├───┤├───────────┤                                             »
q_5: ┤ H ├┤ P(2*x[5]) ├─────────────────────────────────────────────»
     ├───┤├───────────┤                                             »
q_6: ┤ H ├┤ P(2*x[6]) ├─────────────────────────────────────────────»
     ├───┤├───────────┤                                             »
q_7: ┤ H ├┤ P(2*x[7]) ├─────────────────────────────────────────────»
     └───┘└───────────┘                                             »
«                                                  »
«q_0: ────────────────────────────────■─────────■──»
«                                     │         │  »
«q_1: ────────────────────────────────┼────■────┼──»
«     ┌────────────────────────────┐┌─┴─┐┌─┴─┐  │  »
«q_2: ┤ P(2*(π - x[0])*(π - x[2])) ├┤ X ├┤ X ├──┼──»
«     └────────────────────────────┘└───┘└───┘┌─┴─┐»
«q_3: ────────────────────────────────────────┤ X ├»
«                                             └───┘»
«q_4: ─────────────────────────────────────────────»
«                                                  »
«q_5: ─────────────────────────────────────────────»
«                                                  »
«q_6: ─────────────────────────────────────────────»
«                                                  »
«q_7: ─────────────────────────────────────────────»
«                                                  »
«                                                       »
«q_0: ─────────────────────────────────────■─────────■──»
«                                          │         │  »
«q_1: ────────────────────────────────■────┼────■────┼──»
«     ┌────────────────────────────┐┌─┴─┐  │    │    │  »
«q_2: ┤ P(2*(π - x[1])*(π - x[2])) ├┤ X ├──┼────┼────┼──»
«     ├────────────────────────────┤└───┘┌─┴─┐┌─┴─┐  │  »
«q_3: ┤ P(2*(π - x[0])*(π - x[3])) ├─────┤ X ├┤ X ├──┼──»
«     └────────────────────────────┘     └───┘└───┘┌─┴─┐»
«q_4: ─────────────────────────────────────────────┤ X ├»
«                                                  └───┘»
«q_5: ──────────────────────────────────────────────────»
«                                                       »
«q_6: ──────────────────────────────────────────────────»
«                                                       »
«q_7: ──────────────────────────────────────────────────»
«                                                       »
«                                                            »
«q_0: ─────────────────────────────────────■──────────────■──»
«                                          │              │  »
«q_1: ────────────────────────────────■────┼─────────■────┼──»
«                                     │    │         │    │  »
«q_2: ────────────────────────────────┼────┼────■────┼────┼──»
«     ┌────────────────────────────┐┌─┴─┐  │  ┌─┴─┐  │    │  »
«q_3: ┤ P(2*(π - x[1])*(π - x[3])) ├┤ X ├──┼──┤ X ├──┼────┼──»
«     ├────────────────────────────┤└───┘┌─┴─┐└───┘┌─┴─┐  │  »
«q_4: ┤ P(2*(π - x[0])*(π - x[4])) ├─────┤ X ├─────┤ X ├──┼──»
«     └────────────────────────────┘     └───┘     └───┘┌─┴─┐»
«q_5: ──────────────────────────────────────────────────┤ X ├»
«                                                       └───┘»
«q_6: ───────────────────────────────────────────────────────»
«                                             

In [ ]:
class HybridEndToEnd(nn.Module):
    def __init__(self, qnn, feature_dim):
        super().__init__()

        # ---------------------------
        # Classical CNN (VGG19)
        # ---------------------------
        self.cnn = models.vgg19(pretrained=True)

        # Remove VGG classifier
        self.cnn.classifier = nn.Identity()

        # Feature bottleneck → QNN
        self.reduce = nn.Sequential(
            nn.Linear(25088, 128),
            nn.ReLU(),
            nn.Linear(128, feature_dim),
            nn.Tanh()
        )

        # ---------------------------
        # Quantum Layer
        # ---------------------------
        self.qnn = TorchConnector(qnn)

        # Final classifier
        self.fc = nn.Linear(1, 1)

    def forward(self, x):
        x = self.cnn(x)
        x = self.reduce(x)
        x = self.qnn(x)
        return torch.sigmoid(self.fc(x))

In [ ]:
model = HybridEndToEnd(qnn, FEATURE_DIM).to(device)

# Freeze VGG weights for stability
for param in model.cnn.parameters():
    param.requires_grad = False

# Leave bottleneck + QNN trainable

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:
criterion = nn.CrossEntropyLoss()  #this is binary cross-entropy loss
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)

In [ ]:
# for epoch in range(EPOCHS):
#     model.train()
#     total_loss = 0

#     TP = FP = TN = FN = 0

#     for images, labels in train_loader:
#         images = images.to(device)
#         labels = labels.float().view(-1, 1).to(device)

#         optimizer.zero_grad()
#         outputs = model(images)
#         loss = criterion(outputs, labels)

#         loss.backward()
#         optimizer.step()

#         total_loss += loss.item()

#         preds = (outputs >= 0.5).float()

#         TP += ((preds == 1) & (labels == 1)).sum().item()
#         TN += ((preds == 0) & (labels == 0)).sum().item()
#         FP += ((preds == 1) & (labels == 0)).sum().item()
#         FN += ((preds == 0) & (labels == 1)).sum().item()

#     accuracy = 100 * (TP + TN) / (TP + TN + FP + FN)
#     precision = TP / (TP + FP + 1e-8)
#     recall = TP / (TP + FN + 1e-8)

#     print(
#         f"Epoch {epoch+1}/{EPOCHS} | "
#         f"Loss: {total_loss / len(train_loader):.4f} | "
#         f"Acc: {accuracy:.2f}% | "
#         f"Prec: {precision:.3f} | "
#         f"Recall: {recall:.3f}"
#     )
#     torch.save(model,'dermatology_vgg_checkpoint.pt')

In [ ]:
# torch.save(model,'dermatology_vgg.pt')

In [ ]:
# model.eval()
# y_true, y_pred = [], []

# with torch.no_grad():
#     for images, labels in val_loader:
#         images = images.to(device)
#         labels = labels.to(device)

#         preds = model(images)
#         y_pred.extend(preds.cpu().numpy().round().astype(int).flatten())
#         y_true.extend(labels.cpu().numpy().astype(int))

# print("\nClassification Report:\n")
# print(classification_report(y_true, y_pred))

# print("Confusion Matrix:")
# print(confusion_matrix(y_true, y_pred))

In [ ]:
# Unfreeze last convolution block
for param in model.cnn.features[-5:].parameters():
    param.requires_grad = True

optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5)
print("Last VGG block unfrozen")

Last VGG block unfrozen


In [ ]:
criterion = nn.CrossEntropyLoss()  #this is binary cross-entropy loss
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)

In [ ]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    TP = FP = TN = FN = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.float().view(-1, 1).to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = (outputs >= 0.5).float()

        TP += ((preds == 1) & (labels == 1)).sum().item()
        TN += ((preds == 0) & (labels == 0)).sum().item()
        FP += ((preds == 1) & (labels == 0)).sum().item()
        FN += ((preds == 0) & (labels == 1)).sum().item()

    accuracy = 100 * (TP + TN) / (TP + TN + FP + FN)
    precision = TP / (TP + FP + 1e-8)
    recall = TP / (TP + FN + 1e-8)

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Loss: {total_loss / len(train_loader):.4f} | "
        f"Acc: {accuracy:.2f}% | "
        f"Prec: {precision:.3f} | "
        f"Recall: {recall:.3f}"
    )
    torch.save(model,'breastcancer_vgg_checkpoint.pt')

Epoch 1/20 | Loss: 0.0000 | Acc: 75.08% | Prec: 0.000 | Recall: 0.000
Epoch 2/20 | Loss: 0.0000 | Acc: 75.08% | Prec: 0.000 | Recall: 0.000
Epoch 3/20 | Loss: 0.0000 | Acc: 75.08% | Prec: 0.000 | Recall: 0.000
Epoch 4/20 | Loss: 0.0000 | Acc: 75.08% | Prec: 0.000 | Recall: 0.000
Epoch 5/20 | Loss: 0.0000 | Acc: 75.08% | Prec: 0.000 | Recall: 0.000
Epoch 6/20 | Loss: 0.0000 | Acc: 75.08% | Prec: 0.000 | Recall: 0.000
Epoch 7/20 | Loss: 0.0000 | Acc: 75.08% | Prec: 0.000 | Recall: 0.000
Epoch 8/20 | Loss: 0.0000 | Acc: 75.08% | Prec: 0.000 | Recall: 0.000
Epoch 9/20 | Loss: 0.0000 | Acc: 75.08% | Prec: 0.000 | Recall: 0.000
Epoch 10/20 | Loss: 0.0000 | Acc: 75.08% | Prec: 0.000 | Recall: 0.000
Epoch 11/20 | Loss: 0.0000 | Acc: 75.08% | Prec: 0.000 | Recall: 0.000
Epoch 12/20 | Loss: 0.0000 | Acc: 75.08% | Prec: 0.000 | Recall: 0.000
Epoch 13/20 | Loss: 0.0000 | Acc: 75.08% | Prec: 0.000 | Recall: 0.000
Epoch 14/20 | Loss: 0.0000 | Acc: 75.08% | Prec: 0.000 | Recall: 0.000
Epoch 15/20 | L

In [ ]:
torch.save(model,'dermatology_vgg_all_unfrozen.pt')

In [ ]:
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        preds = model(images)
        y_pred.extend(preds.cpu().numpy().round().astype(int).flatten())
        y_true.extend(labels.cpu().numpy().astype(int))

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))



Classification Report:

              precision    recall  f1-score   support

           0       0.68      1.00      0.81       108
           1       0.00      0.00      0.00        52

    accuracy                           0.68       160
   macro avg       0.34      0.50      0.40       160
weighted avg       0.46      0.68      0.54       160

Confusion Matrix:
[[108   0]
 [ 52   0]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
